<a href="https://colab.research.google.com/github/vituhaa/recsys_projects/blob/main/Content_based_pd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Content-based filtering

## Plot description based Recommender

Вычислим попарные показатели сходства для всех фильмов на основе их описаний и будем рекомендовать фильмы на основе этих показателей. Описание сюжета приводится в разделе 'overview' нашего набора данных.

In [1]:
import pandas as pd
import numpy as np

df_credits = pd.read_csv("/content/tmdb_5000_credits.csv")
df_movies = pd.read_csv("/content/tmdb_5000_movies.csv")

In [2]:
df_credits.columns = ['id', 'title', 'cast', 'crew']
df_movies = df_movies.merge(df_credits, on='id')

In [3]:
df_movies['overview'].head(5) # текстовое описание фильмов

,overview
0,"In the 22nd century, a paraplegic Marine is di..."
1,"Captain Barbossa, long believed to be dead, ha..."
2,A cryptic message from Bond’s past sends him o...
3,Following the death of District Attorney Harve...
4,"John Carter is a war-weary, former military ca..."


In [5]:
# используем TF-IDF для каждого overview
from sklearn.feature_extraction.text import TfidfVectorizer

tf_idf = TfidfVectorizer(stop_words='english')
df_movies['overview'] = df_movies['overview'].fillna('')

tf_idf_matrix = tf_idf.fit_transform(df_movies['overview'])

In [6]:
tf_idf_matrix.shape

(4803, 20978)

In [12]:
# cosine similarity
from sklearn.metrics.pairwise import linear_kernel

cosine_sim = linear_kernel(tf_idf_matrix, tf_idf_matrix)

In [14]:
df_movies.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title_x', 'vote_average',
       'vote_count', 'title_y', 'cast', 'crew'],
      dtype='object')

In [16]:
indexes = pd.Series(df_movies.index, index=df_movies['original_title']).drop_duplicates()

In [17]:
def get_recommendations(title, cosine_sim=cosine_sim):
  idx = indexes[title]
  sim_scores = list(enumerate(cosine_sim[idx]))
  sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
  sim_scores = sim_scores[1:11] # топ 10
  movie_idx = [i[0] for i in sim_scores]

  return df_movies['original_title'].iloc[movie_idx]

In [18]:
get_recommendations('The Dark Knight Rises')

,original_title
65,The Dark Knight
299,Batman Forever
428,Batman Returns
1359,Batman
3854,"Batman: The Dark Knight Returns, Part 2"
119,Batman Begins
2507,Slow Burn
9,Batman v Superman: Dawn of Justice
1181,JFK
210,Batman & Robin


## Credits, Genres and Keywords Based Recommender

In [19]:
from ast import literal_eval

features = ['cast', 'crew', 'keywords', 'genres']
for x in features:
  df_movies[x] = df_movies[x].apply(literal_eval)

In [20]:
def get_director(x):
  for i in x:
    if i['job'] == 'Director':
      return i['name']
  return np.nan

In [21]:
def get_list(x):
  if isinstance(x, list):
    names = [i['name'] for i in x]
    if len(names) > 3:
      names = names[:3]
    return names
  return []

In [22]:
df_movies['director'] = df_movies['crew'].apply(get_director)

features = ['cast', 'keywords', 'genres']
for x in features:
  df_movies[x] = df_movies[x].apply(get_list)

In [24]:
df_movies[['original_title', 'cast', 'director', 'keywords', 'genres']].head(3)

,original_title,cast,director,keywords,genres
0,Avatar,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]",James Cameron,"[culture clash, future, space war]","[Action, Adventure, Fantasy]"
1,Pirates of the Caribbean: At World's End,"[Johnny Depp, Orlando Bloom, Keira Knightley]",Gore Verbinski,"[ocean, drug abuse, exotic island]","[Adventure, Fantasy, Action]"
2,Spectre,"[Daniel Craig, Christoph Waltz, Léa Seydoux]",Sam Mendes,"[spy, based on novel, secret agent]","[Action, Adventure, Crime]"


In [25]:
def clean_data(x):
  if isinstance(x, list):
    return [str.lower(i.replace(" ", "")) for i in x]
  else:
    if isinstance(x, str):
      return str.lower(x.replace(" ", ""))
    else:
      return ''

In [27]:
features = ['cast', 'keywords', 'director', 'genres']

for feature in features:
    df_movies[feature] = df_movies[feature].apply(clean_data)

In [28]:
def create_data(x):
    return ' '.join(x['keywords']) + ' ' + ' '.join(x['cast']) + ' ' + x['director'] + ' ' + ' '.join(x['genres'])
df_movies['soup'] = df_movies.apply(create_data, axis=1)

In [29]:
from sklearn.feature_extraction.text import CountVectorizer

count = CountVectorizer(stop_words='english')
count_matrix = count.fit_transform(df_movies['soup'])

In [30]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim_2 = cosine_similarity(count_matrix, count_matrix)

In [31]:
df_movies = df_movies.reset_index()
indices = pd.Series(df_movies.index, index=df_movies['original_title'])

In [33]:
get_recommendations('The Dark Knight Rises', cosine_sim_2)

,original_title
65,The Dark Knight
119,Batman Begins
4638,Amidst the Devil's Wings
1196,The Prestige
3073,Romeo Is Bleeding
3326,Black November
1503,Takers
1986,Faster
303,Catwoman
747,Gangster Squad
